In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2017_2018"
    / "Labs"
)

xpt_file = DATA_DIR / "HDL_J.xpt"
csv_file = DATA_DIR / "HDL_J_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)


#
# Preserve completely untouched pandas-decoded source.
#
# HDL_J requires NO validated numeric repair.
#

source_df = df.copy()


print("=" * 90)
print("NHANES 2017-2018 HDL_J XPT -> CSV")
print("=" * 90)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "LBDHDD",
    "LBDHDDSI"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "HDL_J column names/order differ from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 7435:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 3:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 7,435 rows x 3 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# WHOLE-NUMBER FIELDS
# ------------------------------------------------------------
#
# SEQN
#     Respondent sequence number
#
# LBDHDD
#     Direct HDL cholesterol in mg/dL
#
#
# GENUINE DECIMAL FIELD
# ------------------------------------------------------------
#
# LBDHDDSI
#     Direct HDL cholesterol in mmol/L
#
#
# IMPORTANT:
#
# LBDHDDSI must NOT be converted to Int64.
#
# It must NOT be rounded.
#
# It must NOT receive forced float formatting.
#
# This preserves genuine lab decimals in the same way
# Dietary/Lab/Exam decimals should be preserved throughout
# the broader NHANES pipeline.
#

INTEGER_COLUMNS = [
    "SEQN",
    "LBDHDD"
]


DECIMAL_COLUMNS = [
    "LBDHDDSI"
]


if set(
    INTEGER_COLUMNS
    +
    DECIMAL_COLUMNS
) != set(EXPECTED_COLUMNS):

    raise ValueError(
        "Field type plan does not cover "
        "all HDL_J variables."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Whole-number fields:",
    INTEGER_COLUMNS
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)


# ============================================================
# 5. VERIFY NUMERIC SOURCE STRUCTURE
# ============================================================

numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all three HDL_J variables "
        "to load numerically.\n\n"
        f"Actual numeric columns:\n{numeric_cols}"
    )


print(
    "PASS: All 3 HDL_J variables loaded numerically."
)


# ============================================================
# 6. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known SAS XPORT / pandas zero-decoding artifact:
#
#     5.397605346934028e-79
#
# Never replace this blindly.
#
# Direct inspection of attached HDL_J:
#
#     expected tiny artifacts = 0
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


tiny_counts = (
    df[EXPECTED_COLUMNS]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print(
    "\n--- ORIGINAL XPT TINY-VALUE CHECK ---"
)

print(
    tiny_counts.to_string()
)

print(
    "Total tiny artifacts:",
    tiny_total
)


if tiny_total != 0:

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in HDL_J.\n\n"
        + tiny_counts[
            tiny_counts > 0
        ].to_string()
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 7. CHECK ORIGINAL NUMERIC ZERO PATTERN
# ============================================================

zero_counts = (
    df[EXPECTED_COLUMNS]
    .eq(0)
    .sum()
)


zero_total = int(
    zero_counts.sum()
)


print(
    "\n--- ORIGINAL NUMERIC ZERO CHECK ---"
)

print(
    zero_counts.to_string()
)

print(
    "Total ordinary numeric zeros:",
    zero_total
)


if zero_total != 0:

    raise ValueError(
        "Unexpected numeric zeros found "
        "in attached HDL_J."
    )


print(
    "PASS: Original HDL_J contains no numeric zeros."
)


# ============================================================
# 8. EXPECTED CORRECTED SOURCE
# ============================================================

#
# HDL_J requires NO numeric correction.
#
# Therefore:
#
#     expected_df == source_df
#

expected_df = source_df.copy()


# ============================================================
# 9. DISCOVER FRACTIONAL VALUES
# ============================================================

#
# Run BEFORE converting anything to Int64.
#
# np.round() is used ONLY for comparison.
#
# It DOES NOT alter any data.
#

print(
    "\n--- FRACTIONAL VALUE DISCOVERY ---"
)


fractional_counts = {}


for col in EXPECTED_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if len(values) == 0:

        fractional_count = 0

    else:

        fractional_mask = ~np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )

        fractional_count = int(
            fractional_mask.sum()
        )

    fractional_counts[col] = (
        fractional_count
    )

    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


EXPECTED_FRACTIONAL_COUNTS = {
    "SEQN": 0,
    "LBDHDD": 0,
    "LBDHDDSI": 6737
}


if fractional_counts != EXPECTED_FRACTIONAL_COUNTS:

    raise ValueError(
        "Unexpected fractional-value pattern.\n\n"
        f"Expected:\n{EXPECTED_FRACTIONAL_COUNTS}\n\n"
        f"Actual:\n{fractional_counts}"
    )


print(
    "PASS: LBDHDDSI is the only genuine "
    "decimal variable."
)


# ============================================================
# 10. PROVE INTEGER FIELDS ARE SAFE
# ============================================================

print(
    "\n--- INTEGER FIELD VALIDATION ---"
)


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    whole_mask = np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )

    fractional_count = int(
        (~whole_mask)
        .sum()
    )

    print(
        f"{col}: "
        f"{fractional_count:,} fractional values"
    )

    if fractional_count != 0:

        raise ValueError(
            f"{col} unexpectedly contains "
            "fractional values."
        )


print(
    "PASS: SEQN and LBDHDD are proven "
    "whole-number fields."
)


# ============================================================
# 11. VALIDATE SEQN
# ============================================================

print(
    "\n--- SEQN ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 93705:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 102956:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN 93705-102956 validated."
)


# ============================================================
# 12. CDC CHECK - LBDHDD
# ============================================================

#
# CDC:
#
# Direct HDL cholesterol mg/dL
#
# Range:
#
#     10 - 189
#
# Nonmissing:
#
#     6,738
#
# Missing:
#
#     697
#

hdd_nonmissing = int(
    df["LBDHDD"]
    .notna()
    .sum()
)


hdd_missing = int(
    df["LBDHDD"]
    .isna()
    .sum()
)


hdd_min = float(
    df["LBDHDD"]
    .min()
)


hdd_max = float(
    df["LBDHDD"]
    .max()
)


print(
    "\n--- CDC CHECK: LBDHDD ---"
)

print(
    "Nonmissing:",
    f"{hdd_nonmissing:,}"
)

print(
    "Missing:",
    f"{hdd_missing:,}"
)

print(
    "Minimum:",
    hdd_min
)

print(
    "Maximum:",
    hdd_max
)


if hdd_nonmissing != 6738:

    raise ValueError(
        "LBDHDD nonmissing count differs from CDC."
    )


if hdd_missing != 697:

    raise ValueError(
        "LBDHDD missing count differs from CDC."
    )


if hdd_min != 10:

    raise ValueError(
        "LBDHDD minimum differs from CDC."
    )


if hdd_max != 189:

    raise ValueError(
        "LBDHDD maximum differs from CDC."
    )


print(
    "PASS: LBDHDD matches CDC."
)


# ============================================================
# 13. CDC CHECK - LBDHDDSI
# ============================================================

#
# CDC:
#
# Direct HDL cholesterol mmol/L
#
# Range:
#
#     0.26 - 4.89
#
# Nonmissing:
#
#     6,738
#
# Missing:
#
#     697
#

si_nonmissing = int(
    df["LBDHDDSI"]
    .notna()
    .sum()
)


si_missing = int(
    df["LBDHDDSI"]
    .isna()
    .sum()
)


si_min = float(
    df["LBDHDDSI"]
    .min()
)


si_max = float(
    df["LBDHDDSI"]
    .max()
)


print(
    "\n--- CDC CHECK: LBDHDDSI ---"
)

print(
    "Nonmissing:",
    f"{si_nonmissing:,}"
)

print(
    "Missing:",
    f"{si_missing:,}"
)

print(
    "Minimum:",
    si_min
)

print(
    "Maximum:",
    si_max
)


if si_nonmissing != 6738:

    raise ValueError(
        "LBDHDDSI nonmissing count differs from CDC."
    )


if si_missing != 697:

    raise ValueError(
        "LBDHDDSI missing count differs from CDC."
    )


if not np.isclose(
    si_min,
    0.26,
    rtol=0,
    atol=0
):

    raise ValueError(
        "LBDHDDSI minimum differs from CDC."
    )


if not np.isclose(
    si_max,
    4.89,
    rtol=0,
    atol=0
):

    raise ValueError(
        "LBDHDDSI maximum differs from CDC."
    )


print(
    "PASS: LBDHDDSI matches CDC."
)


# ============================================================
# 14. MISSING-POSITION ALIGNMENT
# ============================================================

#
# LBDHDD and LBDHDDSI represent the same measurement
# in different units.
#
# Their missing positions must therefore match.
#

missing_position_mismatch = int(
    np.sum(
        df["LBDHDD"]
        .isna()
        .to_numpy()
        !=
        df["LBDHDDSI"]
        .isna()
        .to_numpy()
    )
)


print(
    "\n--- LBDHDD / LBDHDDSI MISSING ALIGNMENT ---"
)

print(
    "Missing-position mismatches:",
    missing_position_mismatch
)


if missing_position_mismatch != 0:

    raise ValueError(
        "LBDHDD and LBDHDDSI missing positions differ."
    )


print(
    "PASS: Missing positions match exactly."
)


# ============================================================
# 15. HDL UNIT-CONVERSION CONSISTENCY
# ============================================================

#
# CDC documents:
#
#     LBDHDDSI = LBDHDD * 0.02586
#
# The released SI value is represented to hundredths.
#
# IMPORTANT:
#
# DO NOT recalculate or overwrite LBDHDDSI.
#
# This section is ONLY a consistency check.
#

conversion_df = (
    df[
        [
            "LBDHDD",
            "LBDHDDSI"
        ]
    ]
    .dropna()
)


expected_unrounded_si = (
    conversion_df["LBDHDD"]
    *
    0.02586
)


conversion_difference = (
    conversion_df["LBDHDDSI"]
    -
    expected_unrounded_si
).abs()


max_conversion_difference = float(
    conversion_difference.max()
)


print(
    "\n--- HDL UNIT CONVERSION CHECK ---"
)

print(
    "Records checked:",
    f"{len(conversion_df):,}"
)

print(
    "Maximum absolute difference:",
    max_conversion_difference
)


#
# SI values are represented to hundredths.
#
# Therefore the maximum rounding difference from the
# unrounded conversion should be <= 0.005.
#

if max_conversion_difference > (
    0.005
    +
    1e-12
):

    raise ValueError(
        "LBDHDDSI is inconsistent with "
        "LBDHDD * 0.02586."
    )


print(
    "PASS: LBDHDDSI is consistent with "
    "the documented conversion."
)


# ============================================================
# 16. DECIMAL PRECISION GRID CHECK
# ============================================================

#
# The released LBDHDDSI values occur on a hundredth-unit
# grid.
#
# This is ONLY a check.
#
# It does not round or alter the values.
#

si_values = (
    df["LBDHDDSI"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


hundredth_grid_mask = np.isclose(
    si_values * 100,
    np.round(
        si_values * 100
    ),
    rtol=0,
    atol=1e-10
)


hundredth_grid_failures = int(
    (~hundredth_grid_mask)
    .sum()
)


print(
    "\n--- LBDHDDSI DECIMAL GRID CHECK ---"
)

print(
    "Values not on hundredth grid:",
    hundredth_grid_failures
)


if hundredth_grid_failures != 0:

    raise ValueError(
        "Unexpected LBDHDDSI decimal precision."
    )


print(
    "PASS: All LBDHDDSI values are consistent "
    "with released hundredth precision."
)


# ============================================================
# 17. VERIFY DECIMAL FIELD IS UNCHANGED
# ============================================================

#
# Before converting integer fields, establish that
# LBDHDDSI still exactly equals the original XPT-derived
# values.
#

if not np.allclose(
    source_df["LBDHDDSI"]
    .to_numpy(
        dtype=float
    ),
    df["LBDHDDSI"]
    .to_numpy(
        dtype=float
    ),
    rtol=0,
    atol=0,
    equal_nan=True
):

    raise ValueError(
        "LBDHDDSI changed before integer conversion."
    )


print(
    "PASS: LBDHDDSI remains exactly unchanged."
)


# ============================================================
# 18. CONVERT ONLY PROVEN WHOLE-NUMBER FIELDS
# ============================================================

#
# Convert:
#
#     SEQN
#     LBDHDD
#
# to nullable Int64.
#
#
# DO NOT TOUCH:
#
#     LBDHDDSI
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    whole_mask = np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if not whole_mask.all():

        raise ValueError(
            f"{col} cannot safely be converted "
            "to Int64."
        )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: SEQN and LBDHDD converted "
    "to nullable Int64."
)

print(
    "PASS: LBDHDDSI retained as genuine decimal."
)


# ============================================================
# 19. COMPLETE PRE-EXPORT VALUE FIDELITY
# ============================================================

#
# HDL_J permitted numeric changes:
#
#     NONE
#
# Int64 conversion changes representation only.
#
# Numeric values must remain identical.
#

print(
    "\n--- COMPLETE PRE-EXPORT VALUE FIDELITY ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(pre_export_failures)
    )


print(
    "PASS: All HDL_J numeric values remain unchanged."
)


# ============================================================
# 20. MISSING-VALUE FIDELITY BEFORE EXPORT
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    original_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    current_missing = (
        df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            original_missing
            !=
            current_missing
        )
    )


print(
    "\nMissing-position differences before export:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 21. FINAL ZERO PATTERN BEFORE EXPORT
# ============================================================

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    count = int(
        df[col]
        .eq(0)
        .sum()
    )

    if count > 0:

        final_zero_counts[col] = count


print(
    "\n--- FINAL ZERO PATTERN ---"
)

print(
    final_zero_counts
)


if final_zero_counts != {}:

    raise ValueError(
        "Unexpected zero values introduced:\n"
        + str(final_zero_counts)
    )


print(
    "PASS: No numeric zeros introduced."
)


# ============================================================
# 22. EXPORT CSV
# ============================================================

#
# CRITICAL FIDELITY POLICY
# ------------------------------------------------------------
#
# DO NOT:
#
#     df.round(...)
#
# DO NOT:
#
#     float_format=
#
# DO NOT:
#
#     convert LBDHDDSI to Int64
#
# DO NOT:
#
#     recalculate LBDHDDSI
#
#
# Genuine decimal values are exported using their
# existing floating-point representation.
#
# This is the same rule used to protect genuine
# Dietary, Laboratory, Examination, exposure,
# nutrient, and survey-weight decimal values.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 23. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

#
# IMPORTANT:
#
# For exact numeric fidelity, use:
#
#     float_precision="round_trip"
#
# This avoids false float differences caused by
# reparsing raw decimal strings with pd.to_numeric().
#

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 24. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT XPT -> CSV NUMERIC FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric fidelity differences:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 3 HDL_J variables round-trip "
    "to exact source numeric values."
)


# ============================================================
# 25. CSV MISSING-VALUE FIDELITY
# ============================================================

csv_missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    csv_missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- CSV MISSING-VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    csv_missing_mismatches
)


if csv_missing_mismatches != 0:

    raise ValueError(
        "Missing positions changed during export."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 26. READ FINAL CSV AS RAW TEXT
# ============================================================

#
# Raw text is used ONLY for literal formatting checks.
#
# It is NOT used for exact binary-float equality.
#

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


if raw_csv_df.shape != (
    7435,
    3
):

    raise ValueError(
        "Raw CSV dimensions changed."
    )


if (
    raw_csv_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "Raw CSV column order changed."
    )


# ============================================================
# 27. INTEGER-FIELD .0 CHECK
# ============================================================

#
# ONLY:
#
#     SEQN
#     LBDHDD
#
# are checked for unnecessary ".0".
#
# LBDHDDSI is a genuine decimal variable and is excluded.
#

print(
    "\n--- INTEGER-FIELD .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting in "
        "whole-number fields:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: SEQN and LBDHDD contain no "
    "unnecessary '.0'."
)


# ============================================================
# 28. DECIMAL FIELD RAW-TEXT CHECK
# ============================================================

#
# Do NOT remove ".0" or force formatting in LBDHDDSI.
#
# It is a decimal variable at the VARIABLE level.
#
# One value in this dataset happens to be numerically:
#
#     3.0
#
# That does NOT make the variable an integer field.
#

decimal_tokens = (
    raw_csv_df["LBDHDDSI"]
)


decimal_nonblank = (
    decimal_tokens[
        decimal_tokens != ""
    ]
)


print(
    "\n--- LBDHDDSI RAW DECIMAL CHECK ---"
)

print(
    "Populated decimal tokens:",
    len(decimal_nonblank)
)

print(
    "Tokens ending in '.0' "
    "(allowed for genuine decimal field):",
    int(
        decimal_nonblank
        .str.endswith(".0")
        .sum()
    )
)


if len(decimal_nonblank) != 6738:

    raise ValueError(
        "LBDHDDSI populated CSV count changed."
    )


print(
    "PASS: LBDHDDSI retained as a genuine "
    "decimal field."
)


# ============================================================
# 29. FINAL DECIMAL EXACT FIDELITY
# ============================================================

#
# Use roundtrip_df here.
#
# Do NOT reparse raw text with pd.to_numeric()
# for exact binary-float comparisons.
#

print(
    "\n--- FINAL LBDHDDSI EXACT FIDELITY ---"
)


original_si = (
    expected_df["LBDHDDSI"]
    .to_numpy(
        dtype=float
    )
)


final_si = (
    roundtrip_df["LBDHDDSI"]
    .to_numpy(
        dtype=float
    )
)


si_exact = np.allclose(
    original_si,
    final_si,
    rtol=0,
    atol=0,
    equal_nan=True
)


print(
    "Exact XPT -> CSV -> float round-trip:",
    si_exact
)


if not si_exact:

    raise ValueError(
        "LBDHDDSI genuinely changed "
        "during CSV export."
    )


print(
    "PASS: LBDHDDSI decimal values "
    "preserved exactly."
)


# ============================================================
# 30. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY-VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 31. FINAL ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    count = int(
        values.eq(0)
        .sum()
    )


    if count > 0:

        csv_zero_counts[col] = count


print(
    "\n--- FINAL CSV ZERO PATTERN ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts != {}:

    raise ValueError(
        "Unexpected numeric zeros in final CSV:\n"
        + str(csv_zero_counts)
    )


print(
    "PASS: Final CSV contains no numeric zeros."
)


# ============================================================
# 32. SCIENTIFIC-NOTATION CHECK
# ============================================================

scientific_errors = {}


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]


    count = int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


    if count > 0:

        scientific_errors[col] = count


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    scientific_errors
)


if scientific_errors:

    raise ValueError(
        "Unexpected scientific notation found:\n"
        + str(scientific_errors)
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 33. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 93705:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 102956:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.isna()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains missing SEQN."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN 93705-102956 valid "
    "and contains no '.0'."
)


# ============================================================
# 34. FINAL ALL-COLUMN EXACT FIDELITY
# ============================================================

#
# Use roundtrip_df for exact numeric comparisons.
#

print(
    "\n--- FINAL ALL-COLUMN SOURCE -> CSV FIDELITY ---"
)


final_fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    final_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        final_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        final_fidelity_failures.append(
            col
        )


if final_fidelity_failures:

    raise ValueError(
        "Final source -> CSV fidelity failures:\n"
        + str(final_fidelity_failures)
    )


print(
    "PASS: Final CSV preserves all original "
    "HDL_J numeric values exactly."
)


# ============================================================
# 35. ROUND-TRIP DIFFERENCE DIAGNOSTIC
# ============================================================

#
# Expected:
#
#     maximum difference = 0
#
# for every field.
#

print(
    "\n--- ROUND-TRIP MAXIMUM ABSOLUTE DIFFERENCES ---"
)


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    final_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    valid_mask = (
        ~np.isnan(expected_values)
        &
        ~np.isnan(final_values)
    )


    if valid_mask.any():

        max_difference = float(
            np.max(
                np.abs(
                    expected_values[valid_mask]
                    -
                    final_values[valid_mask]
                )
            )
        )

    else:

        max_difference = 0.0


    print(
        f"{col}: {max_difference}"
    )


# ============================================================
# 36. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)

print(
    "HDL_J CONVERSION COMPLETE"
)

print(
    "=" * 90
)

print(
    "Cycle: 2017-2018"
)

print(
    "Component: Laboratory - "
    "High-Density Lipoprotein Cholesterol"
)

print(
    "Eligible population: "
    "Examined participants age 6+"
)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)

print(
    "Original tiny-value artifacts:",
    tiny_total
)

print(
    "Numeric zero restorations:",
    0
)

print(
    "Other numeric corrections:",
    0
)

print(
    "Whole-number fields:",
    INTEGER_COLUMNS
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "LBDHDD nonmissing:",
    hdd_nonmissing
)

print(
    "LBDHDD range:",
    f"{hdd_min}-{hdd_max}"
)

print(
    "LBDHDDSI nonmissing:",
    si_nonmissing
)

print(
    "LBDHDDSI range:",
    f"{si_min}-{si_max}"
)

print(
    "LBDHDDSI fractional observations:",
    fractional_counts["LBDHDDSI"]
)

print(
    "Maximum conversion-factor difference:",
    max_conversion_difference
)

print(
    "Exact decimal round-trip failures: "
    "0 expected"
)

print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)

print(
    "Missing-position differences: "
    "0 expected"
)

print(
    "Tiny values remaining: "
    "0 expected"
)

print(
    "Unnecessary '.0' in integer fields: "
    "0 expected"
)

print(
    "Scientific notation: "
    "0 expected"
)

print(
    "=" * 90
)

NHANES 2017-2018 HDL_J XPT -> CSV
Rows: 7,435
Columns: 3
PASS: Structure = 7,435 rows x 3 columns.

--- FIELD TYPE PLAN ---
Whole-number fields: ['SEQN', 'LBDHDD']
Genuine decimal fields: ['LBDHDDSI']
PASS: All 3 HDL_J variables loaded numerically.

--- ORIGINAL XPT TINY-VALUE CHECK ---
SEQN        0
LBDHDD      0
LBDHDDSI    0
Total tiny artifacts: 0
PASS: No tiny XPORT artifacts.

--- ORIGINAL NUMERIC ZERO CHECK ---
SEQN        0
LBDHDD      0
LBDHDDSI    0
Total ordinary numeric zeros: 0
PASS: Original HDL_J contains no numeric zeros.

--- FRACTIONAL VALUE DISCOVERY ---
SEQN: 0 fractional observations
LBDHDD: 0 fractional observations
LBDHDDSI: 6,737 fractional observations
PASS: LBDHDDSI is the only genuine decimal variable.

--- INTEGER FIELD VALIDATION ---
SEQN: 0 fractional values
LBDHDD: 0 fractional values
PASS: SEQN and LBDHDD are proven whole-number fields.

--- SEQN ---
Minimum: 93705.0
Maximum: 102956.0
Missing: 0
Duplicates: 0
PASS: SEQN 93705-102956 validated.

--- CDC C